In [1]:
from typing import TypedDict, Annotated

from langchain.chat_models import init_chat_model
from langchain_core.messages import BaseMessage, AIMessage, ToolMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, HumanMessagePromptTemplate
from langchain_core.tools import BaseTool
from langgraph.constants import START, END
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph

from agent_tool import IPLocationByGaoDe, SearchQueryByBoCha
from env_config import load_env_config


ENV_CONFIG = load_env_config()

In [2]:
class MyState(TypedDict):
    prompt_template: ChatPromptTemplate | PromptTemplate
    query: str
    messages: Annotated[list[BaseMessage], add_messages]
    tools: dict[str, BaseTool]


chat_model = init_chat_model(
    model='deepseek-v3.2',
    model_provider='openai',
    temperature=0.1,
    api_key=ENV_CONFIG['openai_api_key'],
    base_url=ENV_CONFIG['openai_base_url'],
    configurable_fields=['model', 'temperature']
)


graph_builder = StateGraph(MyState)

In [3]:
def chatbot(state: MyState):
    llm_with_tools = chat_model.bind_tools(tools=[tool for tool in state.get('tools', {}).values()])

    prompt_template = state.get('prompt_template')
    query = state.get('query')
    chain = prompt_template | llm_with_tools

    resp = chain.invoke({'query': query})

    return {'messages': resp}

graph_builder.add_node('chatbot', chatbot)

In [4]:
def toolcall(state: MyState):
    message = state.get('messages')[-1]
    if not isinstance(message, AIMessage):
        return {'messages': []}

    tool_call_resp = message.tool_calls
    if tool_call_resp is None:
        return {'messages': []}

    messages = []
    for tool_call in tool_call_resp:
        tool_name = tool_call.get('name')
        tool_args = tool_call.get('args')
        tool_id = tool_call.get('id')

        tool = state.get('tools').get(tool_name)
        if tool is None:
            continue

        tool_res = tool.invoke(tool_args)
        tool_msg = ToolMessage(
            content=tool_res,
            tool_call_id=tool_id,
        )
        messages.append(tool_msg)
    return {'messages': messages}

graph_builder.add_node('toolcall', toolcall)

In [5]:
graph_builder.add_edge(START, 'chatbot')
graph_builder.add_edge('chatbot', 'toolcall')
graph_builder.add_edge('toolcall', END)

In [6]:
graph = graph_builder.compile()

In [7]:
prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage('你是一个智能助手，可以帮助用户解决问题，当问题需要工具时，必须一次性调用相应的工具'),
    HumanMessagePromptTemplate.from_template('{query}')
])
query = '查询ip为39.144.46.93的用户位置'

ip_locate_tool = IPLocationByGaoDe()
search_tool = SearchQueryByBoCha()

tool_dict = {
    ip_locate_tool.name: ip_locate_tool,
    search_tool.name: search_tool
}

In [8]:
resp = graph.invoke({
    'query': query,
    'prompt_template': prompt_template,
    'tools': tool_dict,
})